# 🫒 Olive Leaf Disease Classifier — Hackathon CNN
**Target:** MobileNetV3 fine-tuned on olive disease images  
**Classes:** healthy · peacock_eye · anthracnose · verticillium  
**Output:** `best_model.pt` ready for FastAPI inference

## 0. Install & imports

In [ ]:
import os, sys, json, shutil, random, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import MobileNet_V3_Large_Weights
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

## 1. Dataset — auto-detect Kaggle input
This cell finds your olive images wherever Kaggle put them.  
**Supported datasets:**
- `olive-leaf-disease` (dedicated olive dataset)
- Any folder structure with class subfolders

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/kaggle/working/dataset')   # we build this below
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Disease label mapping — habibulbasher01644/olive-leaf-image-dataset
LABEL_MAP = {
    # Kaggle dataset classes
    'Healthy'              : 'healthy',
    'aculus_olearius'      : 'aculus_olearius',      # Anthracnose
    'olive_peacock_spot'   : 'olive_peacock_spot',   # Peacock eye
    # Alternative naming variations
    'healthy'              : 'healthy',
    'Aculus_olearius'      : 'aculus_olearius',
    'Olive_peacock_spot'   : 'olive_peacock_spot',
}

CLASS_NAMES = ['healthy', 'aculus_olearius', 'olive_peacock_spot']
NUM_CLASSES = len(CLASS_NAMES)
print(f'Target classes: {CLASS_NAMES}')

In [ ]:
def find_kaggle_dataset():
    """Find the Kaggle olive dataset in /kaggle/input/"""
    base = Path('/kaggle/input')
    
    # Look for dataset with train/test split already done
    candidates = list(base.glob('*/dataset/train')) + list(base.glob('*/dataset/Test'))
    if candidates:
        return candidates[0].parent.parent  # Returns path to 'dataset' folder
    
    # Fallback: look for any folder with 'train' subfolder
    for p in base.rglob('train'):
        if (p.parent / 'test').exists():
            return p.parent
    
    return None

# ── Run ────────────────────────────────────────────────────────────────────
print('Detecting Kaggle dataset...')
dataset_root = find_kaggle_dataset()

if dataset_root is None:
    print('[WARNING] Dataset not found in /kaggle/input/')
    print('Make sure you added: habibulbasher01644/olive-leaf-image-dataset')
    dataset_root = Path('/kaggle/input/olive-leaf-image-dataset/dataset')

print(f'Using dataset from: {dataset_root}')

# Use dataset directly (already has train/test split)
train_path = dataset_root / 'train'
test_path = dataset_root / 'test'

# Remap folder names to canonical class names using LABEL_MAP
def remap_class_folders(root_path):
    """Remap source folder names to canonical class names."""
    remapped = {}
    for folder in root_path.iterdir():
        if not folder.is_dir():
            continue
        canonical = LABEL_MAP.get(folder.name)
        if canonical is None:
            print(f'  [SKIP] Unknown folder: {folder.name}')
            continue
        remapped[canonical] = folder
    return remapped

print('\nScanning dataset structure...')
train_classes = remap_class_folders(train_path)
test_classes = remap_class_folders(test_path)

print(f'\nTrain classes found: {list(train_classes.keys())}')
print(f'Test classes found:  {list(test_classes.keys())}')

# Verify we have the right classes
expected = set(CLASS_NAMES)
found = set(train_classes.keys()) & set(test_classes.keys())
missing = expected - found

if missing:
    print(f'\n[ERROR] Missing classes: {missing}')
    print('Available classes:')
    for path in train_path.iterdir():
        if path.is_dir():
            img_count = len(list(path.glob('*.jpg')) + list(path.glob('*.png')))
            print(f'  {path.name}: {img_count} images')
else:
    print(f'\n[OK] All {len(CLASS_NAMES)} classes found!')

## 2. Data augmentation — aggressive (real field photos are noisy)

In [ ]:
IMG_SIZE = 224
BATCH    = 32

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Load train and test sets directly from Kaggle dataset
# The test set will be used as validation (since no separate val split)
train_ds = datasets.ImageFolder(train_path, transform=train_tf, 
                                 loader=lambda p: Image.open(p).convert('RGB'))
test_ds  = datasets.ImageFolder(test_path, transform=val_tf,
                                 loader=lambda p: Image.open(p).convert('RGB'))

# Split test into val and test (80/20)
test_size = len(test_ds)
val_size = int(test_size * 0.8)
test_size_split = test_size - val_size
val_ds, test_ds_actual = torch.utils.data.random_split(
    test_ds, [val_size, test_size_split],
    generator=torch.Generator().manual_seed(42)
)

# Weighted sampler for imbalanced training data
targets = [s[1] for s in train_ds.samples]
class_counts = Counter(targets)
weights = [1.0 / class_counts[t] for t in targets]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds_actual, batch_size=BATCH, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds_actual)}')
print(f'Class mapping: {train_ds.class_to_idx}')
print(f'Class counts: {dict(class_counts)}')

# Visualise sample batch
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
for i, ax in enumerate(axes.flat):
    if i < len(imgs):
        img = (imgs[i] * std + mean).clamp(0,1).permute(1,2,0).numpy()
        ax.imshow(img)
        ax.set_title(CLASS_NAMES[labels[i]], fontsize=8)
        ax.axis('off')
plt.suptitle('Augmented training samples')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'augmented_samples.png', dpi=100)
plt.show()
print('[OK] Data loaded successfully!')

## 3. Model — MobileNetV3-Large (fast, mobile-friendly)

In [ ]:
def build_model(num_classes, freeze_backbone=True):
    model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace classifier head
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.Hardswish(),
        nn.Dropout(p=0.4),
        nn.Linear(512, 128),
        nn.Hardswish(),
        nn.Dropout(p=0.3),
        nn.Linear(128, num_classes),
    )
    return model

model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

## 4. Training — Phase 1: head only, then unfreeze

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(imgs)
        correct    += (out.argmax(1) == labels).sum().item()
        n          += len(imgs)
    return total_loss / n, correct / n

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast():
            out  = model(imgs)
            loss = criterion(out, labels)
        total_loss += loss.item() * len(imgs)
        correct    += (out.argmax(1) == labels).sum().item()
        n          += len(imgs)
    return total_loss / n, correct / n

def run_training(model, train_loader, val_loader, epochs, lr,
                 label_smoothing=0.1, tag='phase'):
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.cuda.amp.GradScaler()

    best_val_acc = 0.0
    history = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
        vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
        scheduler.step()
        elapsed = time.time() - t0

        history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
                         'vl_loss': vl_loss, 'vl_acc': vl_acc})

        flag = ''
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pt')
            flag = ' ← best'

        print(f'[{tag}] Ep {epoch:02d}/{epochs}  '
              f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  '
              f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.3f}  '
              f'({elapsed:.0f}s){flag}')

    print(f'\nBest val acc ({tag}): {best_val_acc:.3f}')
    return history

In [ ]:
# ── Phase A: train head only (fast warm-up) ───────────────────────────────
print('=== Phase A: Head only (frozen backbone) ===')
hist_a = run_training(model, train_loader, val_loader,
                      epochs=8, lr=1e-3, tag='head')

In [ ]:
# ── Phase B: unfreeze all layers, fine-tune with low LR ──────────────────
print('=== Phase B: Full fine-tuning (unfrozen backbone) ===')
for param in model.features.parameters():
    param.requires_grad = True

# Load best head weights before fine-tuning
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pt'))

hist_b = run_training(model, train_loader, val_loader,
                      epochs=15, lr=3e-5, tag='full')

## 5. Evaluation — confusion matrix & classification report

In [ ]:
# Load best model
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pt'))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu()
        preds = probs.argmax(1)
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.numpy())

print('Classification Report:')
print(classification_report(all_labels, all_preds,
                             target_names=CLASS_NAMES, digits=3))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Olive Disease CNN')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120)
plt.show()

acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f'\nTest accuracy: {acc:.3f} ({acc*100:.1f}%)')
if acc >= 0.85:
    print('✅ Jury threshold met (≥85%)')
else:
    print('⚠️  Below 85% — see tips below')

In [ ]:
# Training curves
history = hist_a + [{**h, 'epoch': h['epoch'] + len(hist_a)} for h in hist_b]
epochs_arr = [h['epoch'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs_arr, [h['tr_loss'] for h in history], label='Train loss')
ax1.plot(epochs_arr, [h['vl_loss'] for h in history], label='Val loss')
ax1.axvline(len(hist_a) + 0.5, color='gray', linestyle='--', label='Unfreeze')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax1.set_title('Loss curves')

ax2.plot(epochs_arr, [h['tr_acc'] for h in history], label='Train acc')
ax2.plot(epochs_arr, [h['vl_acc'] for h in history], label='Val acc')
ax2.axvline(len(hist_a) + 0.5, color='gray', linestyle='--', label='Unfreeze')
ax2.axhline(0.85, color='green', linestyle=':', label='85% target')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
ax2.set_title('Accuracy curves')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=120)
plt.show()

## 6. Save everything needed for deployment

In [ ]:
# Save class mapping for inference
class_info = {
    'class_names': CLASS_NAMES,
    'class_to_idx': train_ds.class_to_idx,
    'idx_to_class': {v: k for k, v in train_ds.class_to_idx.items()},
    'img_size': IMG_SIZE,
    'mean': [0.485, 0.456, 0.406],
    'std':  [0.229, 0.224, 0.225],
    # Darija labels for the voice response
    'darija_labels': {
        'healthy'            : 'الورقة سليمة، ما فيها حتى مرض. ماتحتاج علاج.',
        'aculus_olearius'    : 'الورقة مصابة بالأكاروس. داء فطري يضر الثمار والأوراق. تحتاج علاج فوري.',
        'olive_peacock_spot' : 'عين الطاووس — داء فطري خطير. تشوف بقع دارة مع حلقة بنية. معدية في الرطوبة.',
    },
    'test_accuracy': float(acc),
}

with open(OUTPUT_DIR / 'class_info.json', 'w', encoding='utf-8') as f:
    json.dump(class_info, f, ensure_ascii=False, indent=2)

print('Saved files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'  {f.name:35s} {size:8.1f} KB')

print('\n[OK] Download best_model.pt and class_info.json')

## 7. Quick inference test — verify before downloading

In [ ]:
def predict_image(img_path, model, class_info, device=DEVICE):
    """Single image inference — returns class, confidence, darija label."""
    tf = transforms.Compose([
        transforms.Resize((class_info['img_size'], class_info['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize(class_info['mean'], class_info['std']),
    ])
    img = Image.open(img_path).convert('RGB')
    tensor = tf(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    idx        = int(probs.argmax())
    cls        = class_info['class_names'][idx]
    confidence = float(probs[idx])
    darija     = class_info['darija_labels'][cls]

    return {
        'class'     : cls,
        'confidence': confidence,
        'darija'    : darija,
        'all_probs' : {class_info['class_names'][i]: float(probs[i])
                       for i in range(len(probs))}
    }

# Test on a few images from test set
test_images = []
for cls in CLASS_NAMES:
    imgs = list((DATA_DIR / 'test' / cls).glob('*.jpg'))[:1] + \
           list((DATA_DIR / 'test' / cls).glob('*.png'))[:1]
    test_images.extend(imgs[:1])

fig, axes = plt.subplots(1, len(test_images), figsize=(4*len(test_images), 4))
if len(test_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_images):
    result = predict_image(img_path, model, class_info)
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    color = 'green' if result['confidence'] > 0.75 else 'orange'
    ax.set_title(
        f"{result['class']}\n{result['confidence']*100:.1f}%\n{result['darija']}",
        fontsize=8, color=color
    )
    ax.axis('off')

plt.suptitle('Inference test — green = confident (>75%), orange = uncertain')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'inference_test.png', dpi=100)
plt.show()
print('\n✅ Model ready. Download output/best_model.pt and output/class_info.json')

## 8. Troubleshooting tips

| Problem | Fix |
|---------|-----|
| Missing class (0 images) | Check folder names vs LABEL_MAP — add the exact folder name |
| Acc stuck < 70% | Unfreeze backbone earlier, increase epochs |
| Val acc drops but train acc rises | Increase Dropout (0.5), add more augmentation |
| CUDA out of memory | Reduce BATCH from 32 → 16 |
| Only 2 olive classes in PlantVillage | Add a Kaggle olive dataset in addition |

**If PlantVillage has very few olive images (<200 per class):**  
Add this cell to download extra data from Kaggle:

In [ ]:
# OPTIONAL: download additional olive dataset if needed
# Run this only if you have few images per class

# Option A: use kagglehub (available in Kaggle notebooks)
# import kagglehub
# path = kagglehub.dataset_download('mohamedalimahjoub/olive-leaf-disease')
# print(path)

# Option B: search and add another dataset via Kaggle UI
# Click "+ Add Data" in the notebook sidebar → search "olive leaf disease"
print('Add your dataset via the Kaggle notebook sidebar: + Add Data')
print('Then re-run cell 1 (build_dataset) to include the new images.')